# Computing Point-in-Time Residual Returns

In this notebook, we will use rolling regressions to compute beta-adjusted (residual) returns for a set of technology stocks in a point-in-time manner suitable for backtesting or live trading.
Residual returns, often referred to as alphas, represent the component of a stock’s return that cannot be explained by its exposure to a benchmark (e.g., an industry ETF). These values are essential for identifying idiosyncratic performance and building market-neutral trading strategies.

We will be performing this analysis in the following few steps
- Downloading of historical data
- Estimate the rolling betas through the use of a lookback window
- Compute the daily residual (Alpha) returns
- Analyse volatility (original vs residual returns)
- Comparing correlations (original vs residual returns)
- Performance ratios (Information and Sharpe)

In [2]:
# Importing the necessary libraries
import pandas as pd
import numpy as np
import yfinance as yf
import warnings
warnings.filterwarnings("ignore")

##### Step 1: Data Collection — Download Historical Prices

We begin by downloading daily close prices for the following tickers from Yahoo Finance, starting from 2016-01-01:


| Stock | Description                                                |
| :---- | :--------------------------------------------------------- |
| META    | Meta Platforms (Facebook)                                  |
| AAPL  | Apple Inc.                                                 |
| AMZN  | Amazon.com Inc.                                            |
| NFLX  | Netflix Inc.                                               |
| GOOGL | Alphabet Inc.                                              |
| QQQ   | Invesco QQQ Trust (NASDAQ 100 ETF) — used as the benchmark |

From these prices, compute daily returns using the adjusted close data:

$$ R_t = \frac{P_t}{P_{t-1}} - 1 $$

where:
- $ P_t $: Adjusted close price at time $ t $
- $ P_{t-1} $: Adjusted close price at time $ t-1 $

In [4]:
tickers = ['META', 'AAPL', 'AMZN', 'NFLX', 'GOOGL', 'QQQ']
data = yf.download(tickers, start='2016-01-01')['Close'] # Retrieving the closing prices
returns = data.pct_change() # Compute the percentage change to get daily returns
returns.head()

[*********************100%***********************]  6 of 6 completed


Ticker,AAPL,AMZN,GOOGL,META,NFLX,QQQ
Date,,,,,,
2016-01-04,NaN,NaN,NaN,NaN,NaN,NaN
2016-01-05,-0.025059,-0.005024,0.002752,0.004989,-0.020917,-0.001735
2016-01-06,-0.019570,-0.001799,-0.002889,0.002336,0.093071,-0.009605
2016-01-07,-0.042204,-0.039058,-0.024140,-0.049043,-0.026513,-0.031314
2016-01-08,0.005288,-0.001464,-0.013617,-0.006025,-0.027671,-0.008201


##### Step 2: Estimating Rolling Betas (252-Day Lookback)

Next, we estimate the beta of each stock relative to **QQQ**, using a **rolling 252-day window** (approximately one trading year). This approach ensures our beta estimates are **point-in-time** and **avoid lookahead bias**. The formula for Beta is as follows:

$$
\beta = \frac{\text{Cov}(R_{\text{stock}}, R_{\text{benchmark}})}{\text{Var}(R_{\text{benchmark}})}
$$

where:  
- $ R_{\text{stock}} $ = returns of the stock  
- $ R_{\text{benchmark}} $ = returns of QQQ  

This measures the **sensitivity** of the stock’s return to movements in the benchmark. For reference, the related concept of **correlation** between a stock and the benchmark is given by:

$$
\text{Corr}(R_{\text{stock}}, R_{\text{benchmark}}) = \frac{\text{Cov}(R_{\text{stock}}, R_{\text{benchmark}})}{\sigma_{R_{\text{stock}}} \cdot \sigma_{R_{\text{benchmark}}}}
$$

where:  
- $ \text{Cov} $ is the covariance,  
- $ \sigma_{R_{\text{stock}}} $ and $ \sigma_{R_{\text{benchmark}}} $ are the standard deviations of returns.

> **Correlation** measures **co-movement**, while **beta** measures **sensitivity and magnitude**.

In [5]:
# Compute the rolling 252-day beta for each stock relative to QQQ
benchmark = returns['QQQ'] # Isolate the benchmark returns
corr = returns.rolling(window=252).corr(benchmark) # Rolling correlation with benchmark
vol = returns.rolling(window=252).std() # Rolling volatility of each stock
beta = corr.multiply(vol, axis=0).divide(vol['QQQ'], axis=0) # Beta calculation
beta.tail()

Ticker,AAPL,AMZN,GOOGL,META,NFLX,QQQ
Date,,,,,,
2026-03-10,1.034352,1.130774,0.880694,1.210832,0.656429,1.0
2026-03-11,1.028098,1.147211,0.870923,1.213431,0.655441,1.0
2026-03-12,1.026796,1.146121,0.871426,1.215952,0.655889,1.0
2026-03-13,1.034708,1.146408,0.870084,1.218064,0.649808,1.0
2026-03-16,1.028935,1.144840,0.865514,1.208725,0.642308,1.0


##### Step 3: Computing Daily Residual (Alpha) Returns

Once we have rolling betas, we can decompose the stock’s return into **expected** and **residual (alpha)** components. Recall that the expected return from benchmark exposure is expressed as follow:
$$
E[R_{\text{stock},t}] = \beta_{\text{stock},t} \cdot R_{\text{benchmark},t}
$$

Given our initial expression for expected returns of a stock being as follow:

$$
R_{\text{stock},t} = \beta_{\text{stock},t} \cdot R_{\text{benchmark},t} + \alpha_{\text{stock},t}
$$

We can rearrange the expression above to get our residual (Alpha) return:
$$
\alpha_{\text{stock},t} = R_{\text{stock},t} - \beta_{\text{stock},t} \cdot R_{\text{benchmark},t}
$$

where:  
- $ R_{\text{stock},t} $ = actual stock return on day $ t $  
- $ \beta_{\text{stock},t} $ = rolling beta (point-in-time)  
- $ R_{\text{benchmark},t} $ = benchmark (QQQ) return on day $ t $  
- $ \alpha_{\text{stock},t} $ = **residual return** (stock-specific performance, *alpha*)  

> These **residuals** represent **stock-specific outperformance or underperformance**, after adjusting for systematic exposure to the market (QQQ).

In [6]:
# Compute the residual returns of our stock
residuals = returns.subtract(beta.multiply(benchmark, axis=0), axis=0)
residuals.tail()

Ticker,AAPL,AMZN,GOOGL,META,NFLX,QQQ
Date,,,,,,
2026-03-10,0.003638,0.003916,0.002205,0.010298,-0.014047,6.098637e-20
2026-03-11,0.000059,-0.007687,0.005521,0.001368,-0.021061,-5.149960e-19
2026-03-12,-0.001740,0.004999,-0.001726,-0.004601,0.005145,-6.591949e-17
2026-03-13,-0.015919,-0.002082,0.000973,-0.031124,0.014455,-2.255141e-17
2026-03-16,0.010515,0.006212,0.003672,0.017940,-0.002937,0.000000e+00


##### Step 4: Analyzing Volatility: Original vs. Residual Returns
Let us compare the volatility (standard deviation) of the raw and residual returns. You should observe that residual returns typically have lower volatility than original returns, since the benchmark-driven (systematic) risk component has been removed.

This demonstrates how much of each stock’s risk was tied to the overall tech sector (via QQQ).

In [7]:
# Compute and compare the volatilities of the residual returns and the original returns
vol = {}
vol['original'] = returns.std()*np.sqrt(252)
vol['residual'] = residuals.std()*np.sqrt(252) 
vol = pd.DataFrame(vol)
vol

,original,residual
Ticker,,
AAPL,0.289937,1.717961e-01
AMZN,0.327569,2.052678e-01
GOOGL,0.287122,1.826811e-01
META,0.384966,2.779169e-01
NFLX,0.419664,3.304010e-01
QQQ,0.222378,5.406183e-16


As shown above, we can see that the residual returns generally have much lower volatility, as we have taken out the 'beta' component which drives much of the stock returns

##### Step 5: Comparing Correlations: Original vs. Residual Returns

Now, let us compute and compare the pairwise correlations of:
- Original stock returns
- Residual (alpha) returns

Expected observation:
> 📉 The correlations between stocks decrease after adjusting for benchmark exposure.

This shows that much of the co-movement among tech stocks is due to shared market/industry factors rather than unique, idiosyncratic behavior.

In [8]:
returns.corr()

Ticker,AAPL,AMZN,GOOGL,META,NFLX,QQQ
Ticker,,,,,,
AAPL,1.000000,0.568462,0.605388,0.517574,0.422396,0.801647
AMZN,0.568462,1.000000,0.634746,0.604789,0.519231,0.764674
GOOGL,0.605388,0.634746,1.000000,0.608728,0.436932,0.782483
META,0.517574,0.604789,0.608728,1.000000,0.450907,0.697249
NFLX,0.422396,0.519231,0.436932,0.450907,1.000000,0.580322
QQQ,0.801647,0.764674,0.782483,0.697249,0.580322,1.000000


In [9]:
residuals.corr()

Ticker,AAPL,AMZN,GOOGL,META,NFLX,QQQ
Ticker,,,,,,
AAPL,1.000000,-0.093507,-0.055736,-0.079815,-0.095648,-0.032098
AMZN,-0.093507,1.000000,0.069858,0.127669,0.133259,0.022717
GOOGL,-0.055736,0.069858,1.000000,0.126533,-0.060206,-0.011288
META,-0.079815,0.127669,0.126533,1.000000,0.078038,0.002694
NFLX,-0.095648,0.133259,-0.060206,0.078038,1.000000,0.002406
QQQ,-0.032098,0.022717,-0.011288,0.002694,0.002406,1.000000


As shown in the above correlation matrix for original returns and residuals (Alpha), the pairwise correlations of the residual returns are generally much lower between stocks. This is because we have taken out one of the major common forces or tides moving these stocks. Another observation is the large drop in correlations with QQQ. The original returns are 0.6 to 0.8 correlated with the QQQ, but the residual returns are almost near 0 correlated with the benchmark QQQ. 

##### Step 6: Performance Ratios: Information Ratio vs. Sharpe Ratio

To evaluate the **risk-adjusted performance** of each stock and its **alpha component**, we compute two key metrics:

1. **Information Ratio (IR)**

$$
\text{IR} = \frac{\text{Mean}(\alpha_{\text{stock}})}{\text{Std}(\alpha_{\text{stock}})}
$$

> This measures the **consistency of alpha generation** relative to **benchmark-adjusted risk** (i.e., residual volatility).  
> A **higher IR** indicates **stronger active management skill** or **idiosyncratic return potential**.

2. **Sharpe Ratio**

$$
\text{Sharpe Ratio} = \frac{\text{Mean}(R_{\text{stock}})}{\text{Std}(R_{\text{stock}})}
$$

> This measures **total risk-adjusted performance**, capturing **both systematic and idiosyncratic** sources of return.

---

**Interpretation & Comparison**
- *Sharpe Ratio*: Focuses on overall return efficiency
- *Information Ratio*: Benchmark-adjusted efficiency (true *active alpha*)

Typically, **IR < Sharpe Ratio**, because removing benchmark exposure reduces **both mean return and volatility**, isolating the smaller **stock-specific alpha**.

In [10]:
df = {}
df['Information Ratio'] = residuals.mean() / residuals.std() * np.sqrt(252)
df['Sharpe Ratio'] = returns.mean() / returns.std() * np.sqrt(252)
df = pd.DataFrame(df)
df

,Information Ratio,Sharpe Ratio
Ticker,,
AAPL,0.352667,0.947311
AMZN,0.011369,0.728237
GOOGL,0.275656,0.858316
META,0.053746,0.658500
NFLX,0.241719,0.719349
QQQ,0.533962,0.891113


#### Performance Analysis

The above table presents the **Information Ratio (IR)** and **Sharpe Ratio** for several large-cap technology stocks—Apple (AAPL), Amazon (AMZN), Alphabet (GOOGL), Meta (META), and Netflix (NFLX)—as well as the ETF QQQ. These two metrics measure risk-adjusted performance but capture slightly different dimensions. The Sharpe Ratio evaluates excess return relative to total volatility, while the Information Ratio measures excess return relative to a benchmark, scaled by tracking error.

Overall, QQQ demonstrates the strongest benchmark-relative performance, with the highest Information Ratio of 0.534. This suggests that the ETF delivered the most consistent outperformance relative to its benchmark. Its Sharpe Ratio of 0.891 is also relatively high, indicating strong returns relative to total risk. Among the individual stocks, Apple (AAPL) stands out with the highest Sharpe Ratio of 0.947, implying that it generated the best return per unit of volatility. Apple also shows a relatively strong Information Ratio of 0.353, suggesting that its performance was both strong and consistent relative to the benchmark.

Alphabet (GOOGL) also performs well across both metrics, with an Information Ratio of 0.276 and a Sharpe Ratio of 0.858. These values indicate solid and stable risk-adjusted returns, though slightly below those of Apple and QQQ. Netflix (NFLX) exhibits moderate performance, with an Information Ratio of 0.242 and a Sharpe Ratio of 0.719. While its Sharpe Ratio suggests acceptable returns relative to volatility, its lower Information Ratio indicates less consistent outperformance relative to the benchmark.

Meta (META) shows weaker results compared to the other assets, with an Information Ratio of 0.054 and a Sharpe Ratio of 0.659. Although returns appear positive on a risk-adjusted basis, the low Information Ratio suggests limited ability to consistently outperform the benchmark. Amazon (AMZN) has the lowest Information Ratio at 0.011, indicating almost no excess return relative to the benchmark despite having a moderate Sharpe Ratio of 0.728. This suggests that while Amazon produced reasonable returns relative to volatility, it did not significantly outperform the benchmark during the period analyzed.

In summary, QQQ provides the strongest benchmark-relative performance, Apple delivers the best overall volatility-adjusted returns, and Alphabet shows consistently strong results across both measures. Netflix performs moderately, while Meta and Amazon lag behind in terms of benchmark-relative performance.